# Environment Variables

CIM-Graph is configured with `CIMG_*` environment variables. Set them in the shell or with `os.environ` before creating a connection. Every connection reads the profile, namespace, identifier style, units, log level and connection settings as they are when it is created, so a change applies to the next connection, not to existing ones. [example.env](https://github.com/PNNL-CIM-Tools/CIM-Graph/blob/main/example.env) has sample values.

The getter functions are in `cimgraph.core`. Most cache their value: call `.cache_clear()` on a getter to re-read the variable outside a connection.

### Model settings

| Variable | Default | Purpose |
|---|---|---|
| `CIMG_CIM_PROFILE` | none (required) | CIM profile (see below) |
| `CIMG_NAMESPACE` | `http://iec.ch/TC57/CIM100#` | Namespace of the `cim:` prefix |
| `CIMG_IEC61970_552` | `552-NEW` | Identifier serialization style: `552-NEW`, `552-LEGACY` or `CGMES` |
| `CIMG_USE_UNITS` | `true` | Read unit values as `CIMUnit`s; `false` reads plain floats in base SI units |
| `CIMG_VALIDATION_LOG_LEVEL` | `WARNING` | Log level of profile validation messages (e.g. a class not in the profile) |
| `CIMG_MODEL_AUTHORITY` | `https://github.com/PNNL-CIM-Tools/CIM-Graph` | Model authority URI: the namespace of seeded UUIDs, written in file headers |
| `CIMG_MODEL_NAME` | empty | Model name within the authority, mixed into seeded UUIDs |
| `CIMG_SYSTEM_MVA_BASE` | `100` | System MVA base of a graph model whose graph has no `BasePower` |
| `CIMG_ALLOW_UNDEFINED_ATTRIBUTES` | `false` | Keep properties the profile does not define, as text on the object (e.g. `line.vendorRating == '42.5'`), instead of logging and dropping them. They are not typed (a reference or enumeration is its identifier text) and are not written by `write_xml`/`write_json_ld` |
| `CIMG_ALLOW_REMOTE_CONTEXT` | `false` | Let `JSONLDFile` fetch JSON-LD contexts and unit vocabularies that are not in the cimprofiles offline cache |
| `CIMG_AUTHORITY_FROM_IS_VERSION_OF` | `false` | Take the model authority of a `dcat:Dataset` file header that has no `md:Model.modelingAuthoritySet` from its `dcat:isVersionOf`, as CGMES files (Nordic44) carry it. Otherwise `isVersionOf` is a model series and the model scope is left unchanged, with a warning |

### Connection settings

| Variable | Default | Read by |
|---|---|---|
| `CIMG_URL` | `http://localhost:8889/bigdata/namespace/kb/sparql` (with a warning) | `BlazegraphConnection`, `GraphDBConnection`, `Neo4jConnection` |
| `CIMG_HOST` | `localhost` | `MySQLConnection`, `GridappsdConnection` |
| `CIMG_PORT` | `61613` | `MySQLConnection`, `GridappsdConnection` |
| `CIMG_DATABASE` | `powergridmodel` | `Neo4jConnection`, `MySQLConnection`, `GridappsdConnection` |
| `CIMG_USERNAME` | none: raises `ValueError` | `Neo4jConnection`, `MySQLConnection`, `GridappsdConnection` |
| `CIMG_PASSWORD` | none: raises `ValueError` | `Neo4jConnection`, `MySQLConnection`, `GridappsdConnection` |

`XMLFile`, `JSONLDFile` and `RDFlibConnection` take a file name and read none of the connection settings. With the `552-LEGACY` and `CGMES` styles, the SPARQL and Cypher queries also use `CIMG_URL` as the base IRI that `rdf:ID` values resolve against.

----

## CIMG_CIM_PROFILE

The CIM profile whose classes the connections create. A comma-separated list of profiles is merged into one profile at runtime. Each name is resolved in this order:

1. A pre-0.6 `cimgraph.data_profile.<name>` path resolves as `<name>` does, with a `DeprecationWarning`.
2. A name whose first part is a known profile in `cimprofiles.PROFILE_PACKAGES`, such as `cimhub_2023` or `cim18gmdm.connectivity`, resolves to its `cimprofiles` module.
3. Any other dotted module path whose top-level package is importable is used as given, such as `cimprofiles.cim17.cimhub_2023` or `my_package.my_profile`. A custom profile package outside `cimprofiles` works this way.
4. Anything else raises `ValueError`.

Every spelling of a profile gives the same module.

`get_cim_profile()` returns the value and the profile module:

In [1]:
import os
from cimgraph.core import get_cim_profile

os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
get_cim_profile.cache_clear()
cim_profile, cim = get_cim_profile()
print(cim_profile, '->', cim.__name__)

cimhub_2023 -> cimprofiles.cim17.cimhub_2023


In [2]:
os.environ['CIMG_CIM_PROFILE'] = 'cim18gmdm.connectivity,cim18gmdm.electrical'
get_cim_profile.cache_clear()
cim_profile, cim = get_cim_profile()
print(cim.__name__)

merged_cim_profile:cimprofiles.cim18.cim18gmdm.connectivity,cimprofiles.cim18.cim18gmdm.electrical


A name that is neither a known profile nor an importable module path raises a `ValueError` that lists the known profiles:

In [3]:
os.environ['CIMG_CIM_PROFILE'] = 'cim99'
get_cim_profile.cache_clear()
try:
    get_cim_profile()
except ValueError as error:
    print(error)

Unknown CIM profile 'cim99': not an importable module path, nor one of the known profiles: cim16v33, cgmes_2_4_0, cgmes_3_0_0, cim17v40, cimhub_2023, rc4_2021, cim18gad, cim18gmdm, cim18v17, cim18v17_distribution, cim18v17_transmission, cimhub_2026


----

## CIMG_NAMESPACE

The namespace tied to the `cim:` prefix. Each CIM version uses its own, which is declared at the top of a CIM XML file, e.g. `xmlns:cim="http://iec.ch/TC57/CIM100#"`. The default is the CIM100 namespace used by CIM 17.

In [4]:
from cimgraph.core import get_namespace

os.environ.pop('CIMG_NAMESPACE', None)
get_namespace.cache_clear()
print(get_namespace())

os.environ['CIMG_NAMESPACE'] = 'http://iec.ch/TC57/2011/CIM-schema-cim15#'
get_namespace.cache_clear()
print(get_namespace())

http://iec.ch/TC57/CIM100#
http://iec.ch/TC57/2011/CIM-schema-cim15#


----

## CIMG_IEC61970_552

The serialization style of object identifiers in files and database queries:

| Style | rdf:ID / rdf:about | References | mRID (when it is the UUID) |
|---|---|---|---|
| `552-NEW` (default) | `rdf:about="urn:uuid:<uuid>"` | `urn:uuid:<uuid>` | `<uuid>` |
| `552-LEGACY` | `rdf:ID="_<uuid>"` | `#_<uuid>` | `_<uuid>` |
| `CGMES` | `rdf:ID="_<uuid>"` | `#_<uuid>` | `<uuid>` |

`552-NEW` (IEC 61970-552 edition 2):

```xml
<cim:Breaker rdf:about="urn:uuid:52de9189-20dc-4c73-bdee-e960fe1f9493">
  <cim:Equipment.EquipmentContainer rdf:resource="urn:uuid:49ad8e07-3bf9-a4e2-cb8f-c3722f837b62"/>
</cim:Breaker>
```

`552-LEGACY` (edition 1):

```xml
<cim:Breaker rdf:ID="_52de9189-20dc-4c73-bdee-e960fe1f9493">
  <cim:Equipment.EquipmentContainer rdf:resource="#_49ad8e07-3bf9-a4e2-cb8f-c3722f837b62"/>
</cim:Breaker>
```

The file readers recognize every style when reading. The setting chooses how identifiers are written and queried. The older integer `CIMG_IEC61970_301` (8 for `552-NEW`, 7 or less for `552-LEGACY`) is still read when `CIMG_IEC61970_552` is not set, with a `DeprecationWarning`.

In [5]:
from cimgraph.core import get_iec61970_552

os.environ['CIMG_IEC61970_552'] = '552-LEGACY'
get_iec61970_552.cache_clear()
print(get_iec61970_552())

552-LEGACY


----

## CIMG_USE_UNITS

With the default `true`, readers set unit attributes to `CIMUnit`s, which convert with `.to()`. With `false`, they are plain floats in the base SI unit. A connection reads the setting when it is created:

In [6]:
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
os.environ['CIMG_NAMESPACE'] = 'http://iec.ch/TC57/CIM100#'
os.environ['CIMG_IEC61970_552'] = '552-NEW'
import cimprofiles.cim17.cimhub_2023 as cim

from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel

feeder = cim.Feeder(mRID='49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
for use_units in ['true', 'false']:
    os.environ['CIMG_USE_UNITS'] = use_units
    network = FeederModel(container=feeder,
                          connection=XMLFile('../../tests/test_models/ieee13.xml'),
                          distributed=False)
    line = network.first(cim.ACLineSegment)
    print(f'CIMG_USE_UNITS={use_units}: {line.length!r}')

print(f'{line.name} is {cim.Length(line.length).to("ft"):.0f} ft long')

CIMG_USE_UNITS=true: 91.44 meter
CIMG_USE_UNITS=false: 91.44
645646 is 300 ft long


----

## CIMG_MODEL_AUTHORITY and CIMG_MODEL_NAME

The model scope. UUIDs seeded from names and non-UUID mRIDs are version 5 UUIDs in the namespace of the model authority, so the same name under different authorities gives different UUIDs. Set the authority to your organization's URI. The default is the same on every machine.

The file readers set the scope from each file's header (`md:FullModel` in CIM XML, `dcat:Dataset` in JSON-LD). `set_model_scope(authority, model_name)` sets both variables and clears the getters' caches:

In [7]:
from cimgraph.core import get_model_authority, get_model_name, set_model_scope

set_model_scope('https://utility.example.com/models', 'feeder_12')
print(get_model_authority(), get_model_name())

https://utility.example.com/models feeder_12
